In [ ]:
import os
import asyncio
import time
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, function_tool, OpenAIChatCompletionsModel, input_guardrail, GuardrailFunctionOutput, InputGuardrailTripwireTriggered
from pydantic import BaseModel, Field
import httpx
import gradio as gr

API_KEY = os.getenv("OPENROUTER_API_KEY")
TELEGRAM_BOT_TOKEN = os.getenv("TELEGRAM_BOT_TOKEN")
TELEGRAM_CHAT_ID = os.getenv("TELEGRAM_CHAT_ID")
MODEL = "openai/gpt-4o"
MAX_HANDOFFS = 10

In [ ]:
load_dotenv()

client = AsyncOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=API_KEY,
)
openrouter_model = OpenAIChatCompletionsModel(model=MODEL, openai_client=client)

In [ ]:
TELEGRAM_TIMEOUT = 25.0
TELEGRAM_MAX_LENGTH = 4096

async def send_telegram(message):
    if not TELEGRAM_BOT_TOKEN or not TELEGRAM_CHAT_ID:
        return
    if len(message) > TELEGRAM_MAX_LENGTH:
        message = message[: TELEGRAM_MAX_LENGTH - 4] + "..."
    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
    try:
        async with httpx.AsyncClient(timeout=TELEGRAM_TIMEOUT) as http:
            r = await http.post(url, json={"chat_id": TELEGRAM_CHAT_ID, "text": message})
            r.raise_for_status()
    except httpx.ConnectTimeout:
        print("Telegram notification failed: connection timed out (check network/firewall or api.telegram.org access).")
    except Exception as e:
        err = str(e) or repr(e)
        print(f"Telegram notification failed: {err}")

In [ ]:
class QueryCheck(BaseModel):
    is_valid: bool = Field(description="True if the query is suitable for research.")
    reason: str = Field(description="Brief reason if not valid.")

query_check_agent = Agent(
    name="QueryCheck",
    instructions="Check if the user query is valid for research: not empty, at least a few words, and a real question.",
    output_type=QueryCheck,
    model=openrouter_model,
)

@input_guardrail
async def guardrail_query(ctx, agent, message):
    result = await Runner.run(query_check_agent, message, context=ctx.context)
    return GuardrailFunctionOutput(
        output_info={"query_check": result.final_output},
        tripwire_triggered=not result.final_output.is_valid,
    )

class SubQuestions(BaseModel):
    sub_questions: list[str] = Field(description="Three to five specific sub-questions as a list of strings.")

planner_agent = Agent(
    name="Planner",
    instructions="Break the user's question into 3 to 5 specific sub-questions. Return only the list of sub-questions.",
    model=openrouter_model,
    output_type=SubQuestions,
    input_guardrails=[guardrail_query],
)

async def run_planner(question):
    start = time.perf_counter()
    result = await Runner.run(planner_agent, question)
    duration = time.perf_counter() - start
    return result.final_output.sub_questions[:5], duration

In [ ]:
class ResearchAnswer(BaseModel):
    text: str = Field(description="Concise factual answer to the sub-question.")

research_worker_agent = Agent(
    name="ResearchWorker",
    instructions="Answer the given sub-question in a concise, factual way. Use a short paragraph or bullet points.",
    model=openrouter_model,
    output_type=ResearchAnswer,
)

async def run_single_worker(sub_question):
    start = time.perf_counter()
    result = await Runner.run(research_worker_agent, sub_question)
    duration = time.perf_counter() - start
    return result.final_output.text, duration

async def run_research_workers(sub_questions):
    start = time.perf_counter()
    tasks = [run_single_worker(q) for q in sub_questions]
    results = await asyncio.gather(*tasks)
    outputs = [r[0] for r in results]
    duration = time.perf_counter() - start
    return outputs, duration

In [ ]:
@function_tool
def format_section(title: str, body: str) -> str:
    return f"## {title}\n\n{body}"

class ReportOutput(BaseModel):
    markdown_report: str = Field(description="The final combined report in markdown.")

synthesizer_agent = Agent(
    name="Synthesizer",
    instructions="Combine the research excerpts into one final report. Resolve overlaps and present in logical flow with clear sections. You may use format_section to build section headings. Output markdown.",
    model=openrouter_model,
    output_type=ReportOutput,
    tools=[format_section],
)

async def run_synthesizer(worker_outputs, original_question):
    start = time.perf_counter()
    combined = "\n\n---\n\n".join(worker_outputs)
    user_input = f"Original question: {original_question}\n\nResearch excerpts:\n{combined}"
    result = await Runner.run(synthesizer_agent, user_input)
    duration = time.perf_counter() - start
    return result.final_output.markdown_report, duration

In [ ]:
class TelegramMessage(BaseModel):
    message: str = Field(description="Single message for Telegram, under 4000 characters.")

telegram_formatter_agent = Agent(
    name="TelegramFormatter",
    instructions=(
        "You format research results for one Telegram message. Rules:\n"
        "- Plain text only: no ** or ## or markdown. Use CAPS or short labels for headings.\n"
        "- Summarize the report into a short notification: state the question, then 3-6 bullet takeaways (one line each). Do not paste the full report.\n"
        "- Use double newline between sections, single newline for list items.\n"
        "- Keep under 4000 characters. If input is an error message, output it clearly in one short paragraph."
    ),
    model=openrouter_model,
    output_type=TelegramMessage,
)

async def format_for_telegram(question: str, content: str) -> str:
    result = await Runner.run(
        telegram_formatter_agent,
        f"Question: {question}\n\nContent to format for Telegram:\n{content}",
    )
    return result.final_output.message

In [ ]:
async def handoff(state, agent_fn, input_data):
    if state["handoff_count"] >= MAX_HANDOFFS:
        return None, "Handoff limit reached."
    if isinstance(input_data, tuple):
        result = await agent_fn(*input_data)
    else:
        result = await agent_fn(input_data)
    state["handoff_count"] += 1
    return result, None

In [ ]:
async def orchestrator(question):
    state = {"handoff_count": 0}
    plan_result, err = await handoff(state, run_planner, question)
    if err:
        return err, None
    sub_questions = plan_result[0] if isinstance(plan_result, tuple) else plan_result
    if not isinstance(sub_questions, list):
        sub_questions = [sub_questions]
    workers_result, err = await handoff(state, run_research_workers, sub_questions)
    if err:
        return err, None
    worker_outputs = workers_result[0] if isinstance(workers_result, tuple) else workers_result
    report_result, err = await handoff(state, run_synthesizer, (worker_outputs, question))
    if err:
        return err, None
    report = report_result[0] if isinstance(report_result, tuple) else report_result
    return report, None

In [ ]:
def run_research(question, history):
    if not question or not question.strip():
        return {"content": "", "files": []}
    try:
        report, _ = asyncio.run(orchestrator(question.strip()))
    except InputGuardrailTripwireTriggered as e:
        try:
            query_check = e.guardrail_result.output.output_info.get("query_check")
            reason = getattr(query_check, "reason", None) or "Query did not pass the research check."
        except Exception:
            reason = "Your question was not suitable for research (e.g. too short, empty, or not a real question)."
        msg = (
            "**Research guardrail**\n\n"
            f"Your query could not be processed: {reason}\n\n"
            "Please try a clearer, longer research question (at least a few words)."
        )
        return {"content": msg, "files": []}
    except Exception as e:
        print(f"Error: {e}")
        return {"content": str(e), "files": []}
    display = report if isinstance(report, str) else str(report)
    try:
        async def _send_one_telegram():
            formatted = await format_for_telegram(question.strip(), display)
            await send_telegram(formatted)
        asyncio.run(_send_one_telegram())
    except Exception as e:
        print(f"Telegram send failed: {e}")
    return {"content": display, "files": []}

demo = gr.ChatInterface(
    fn=run_research,
    title="Deep Researcher",
    description="Enter a research question. The planner will break it into sub-tasks, workers will research in parallel, and the synthesizer will produce a final report.",
)
demo.launch(share=True)